# Text Classification, Sentiment, and Topic Modeling

## MSA 8700 — Module 8: NLP and Text Processing

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **TF-IDF** representation and why it works for text classification
2. Build a **supervised text classification** pipeline with scikit-learn (vectorize → train → predict)
3. Evaluate classifier performance with **accuracy, precision, recall, and F1**
4. Perform **lexicon-based sentiment analysis** with VADER and interpret its output
5. Apply **Latent Dirichlet Allocation (LDA)** for unsupervised topic discovery
6. Understand how these classical techniques fit into **agentic AI workflows** alongside LLMs

---

# Part 1: Why Classical Text Classification?

## When to Use Classical Models Instead of LLMs

LLMs can classify text in-context — just ask "Is this review positive or negative?" — but classical models remain attractive for many operational tasks:

| Scenario | Classical Models | LLMs |
|----------|-----------------|------|
| **Labeled data with stable categories** | Train once, run forever | Prompt engineering per query |
| **Predictable latency** | Sub-millisecond inference | Seconds per call |
| **Cost at scale** | Free after training | Per-token charges |
| **On-prem / constrained environments** | Small model files (~MB) | Requires API or large GPU |
| **Reproducibility** | Deterministic for fixed model | Stochastic outputs |
| **Thousands of documents per second** | Easily achievable | Not feasible at API prices |

### The Agentic Pattern

```
Incoming ticket / message
          ↓
Classical classifier: coarse routing (billing / technical / shipping)
          ↓
LLM agent: draft response, summarize, handle nuanced cases
```

The classical model handles the **high-volume triage** (cheap, fast, deterministic). The LLM handles the **high-value reasoning** (drafting, summarizing, escalation).

## Setup

In [ ]:
# Install if needed:
# pip install scikit-learn nltk

import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("Setup complete.")

---

# Part 2: From Text to Features — TF-IDF

## The Problem: Machine Learning Needs Numbers

Classifiers like Logistic Regression and SVM operate on **numerical feature vectors**, not raw text. We need a way to convert text into numbers.

## Bag of Words (BoW)

The simplest approach: count how many times each word appears in a document.

```
Document: "the delivery was late and the package was damaged"

Vocabulary:   [and, damaged, delivery, late, package, the, was]
BoW vector:   [ 1,    1,       1,      1,     1,      2,   2 ]
```

**Problem**: Common words like "the" and "was" get high counts but carry little meaning.

## TF-IDF: Term Frequency × Inverse Document Frequency

**TF-IDF** solves this by weighing words that are frequent in a document but rare across the corpus:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Where:
- **TF(t, d)**: How often term *t* appears in document *d* (frequency within the document)
- **IDF(t)**: How rare term *t* is across all documents: $\log\frac{N}{\text{df}(t)}$ where *N* = total documents, *df(t)* = documents containing *t*

**Effect**: Words that appear in many documents ("the", "is", "was") get low IDF. Words unique to a few documents ("damaged", "excellent") get high IDF. This gives meaningful words more weight.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# A small corpus to demonstrate TF-IDF
documents = [
    "The delivery was late and the package was damaged.",
    "Great service, the support team was very helpful.",
    "The product quality is terrible and customer service was awful.",
    "Fast delivery, excellent packaging, very satisfied.",
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

# Display as a DataFrame for readability
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(X.toarray(), columns=feature_names,
                  index=[f"Doc {i+1}" for i in range(len(documents))])

print("TF-IDF Matrix (each row is a document, each column is a word):\n")
# Show only non-zero columns for readability
print(df.round(2).to_string())

In [ ]:
# Which words have the highest TF-IDF scores per document?
print("Top 3 most important words per document:\n")
for i, doc_text in enumerate(documents):
    row = df.iloc[i]
    top_words = row.nlargest(3)
    words_str = ", ".join(f"{word} ({score:.2f})" for word, score in top_words.items())
    print(f"  Doc {i+1}: {words_str}")
    print(f"          \"{doc_text[:60]}...\"\n")

Notice how common words like "the" and "was" get low scores, while distinctive words like "damaged", "helpful", "terrible", "satisfied" get high scores. This is exactly what makes TF-IDF effective for classification.

---

# Part 3: Supervised Text Classification

## The Pipeline

```
Step 1: Preprocess — tokenize, lowercase, remove stopwords
            ↓
Step 2: Vectorize — convert to TF-IDF features
            ↓
Step 3: Train — fit a classifier (Logistic Regression, SVM, etc.)
            ↓
Step 4: Predict — classify new, unseen documents
```

## Example 1: Sentiment Classification (Positive / Negative)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Training data — labeled examples
texts = [
    "Delivery was late and the package was damaged.",
    "Terrible quality, the product broke after one day.",
    "Very disappointed with the customer service.",
    "The item never arrived and no one responded to my emails.",
    "Worst purchase I have ever made.",
    "Great service, the support team was very helpful.",
    "Excellent product, exactly what I needed.",
    "Fast shipping and wonderful packaging.",
    "I love this product, will definitely buy again.",
    "Outstanding quality and the price was very reasonable.",
]
labels = ["negative", "negative", "negative", "negative", "negative",
          "positive", "positive", "positive", "positive", "positive"]

# Step 1-2: Vectorize with TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X_train = vectorizer.fit_transform(texts)

# Step 3: Train a Logistic Regression classifier
clf = LogisticRegression()
clf.fit(X_train, labels)

print(f"Training set size: {len(texts)} documents")
print(f"Vocabulary size:   {len(vectorizer.get_feature_names_out())} features")
print(f"Model:             Logistic Regression")

In [ ]:
# Step 4: Predict on new, unseen text
new_texts = [
    "The product quality is terrible.",
    "Amazing experience, highly recommended!",
    "The package arrived damaged and late.",
    "Best customer service I have ever received.",
    "Not worth the money at all.",
]

X_new = vectorizer.transform(new_texts)
predictions = clf.predict(X_new)
probabilities = clf.predict_proba(X_new)

print(f"{'Text':<50} {'Prediction':<12} {'Confidence'}")
print("-" * 75)
for text, pred, proba in zip(new_texts, predictions, probabilities):
    confidence = max(proba) * 100
    print(f"{text:<50} {pred:<12} {confidence:.1f}%")

### What the Classifier Learned

We can inspect the model's coefficients to see which words it associates most strongly with each class.

In [ ]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
coefficients = clf.coef_[0]

# Sort features by coefficient
sorted_indices = np.argsort(coefficients)

# Top negative indicators (most negative coefficients)
print("Words most associated with NEGATIVE:")
for idx in sorted_indices[:8]:
    print(f"  {feature_names[idx]:<18} {coefficients[idx]:+.3f}")

print("\nWords most associated with POSITIVE:")
for idx in sorted_indices[-8:][::-1]:
    print(f"  {feature_names[idx]:<18} {coefficients[idx]:+.3f}")

This transparency is a major advantage of classical models: you can **explain exactly why** a document was classified a certain way, by looking at which words had the strongest influence.

## Example 2: Ticket Triage — Multi-Class Classification

A more realistic scenario: a **ticket triage agent** routes incoming support tickets to the correct team.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Training data: customer support tickets with category labels
ticket_texts = [
    # Billing
    "I was charged twice for the same order.",
    "My credit card was billed the wrong amount.",
    "I need a refund for order number 4521.",
    "The invoice does not match my purchase.",
    "Why was I charged after canceling my subscription?",
    "I see an unexpected charge on my statement.",
    "Please update my payment method on file.",
    "The promo code did not apply the discount.",
    # Technical
    "The app crashes every time I open it.",
    "I cannot log in to my account.",
    "The website is showing a 500 error.",
    "My password reset link is not working.",
    "The search feature returns no results.",
    "I get an error message when uploading files.",
    "The page loads very slowly on mobile.",
    "My account settings are not saving properly.",
    # Shipping
    "My package has not arrived yet.",
    "The delivery was delayed by two weeks.",
    "I received the wrong item in my order.",
    "The package arrived damaged.",
    "Can I change the shipping address for my order?",
    "I need to track my shipment.",
    "The delivery driver left the package in the rain.",
    "My order says delivered but I never received it.",
]

ticket_labels = (
    ["billing"] * 8 +
    ["technical"] * 8 +
    ["shipping"] * 8
)

# Split into train and test sets
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    ticket_texts, ticket_labels, test_size=0.25, random_state=42, stratify=ticket_labels
)

# Build the pipeline
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_txt)
X_test = vectorizer.transform(X_test_txt)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)

print("=== Ticket Triage Classifier — Evaluation ===")
print(f"Training set: {len(X_train_txt)} tickets")
print(f"Test set:     {len(X_test_txt)} tickets")
print(f"Features:     {X_train.shape[1]} (unigrams + bigrams)\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Classify new incoming tickets
new_tickets = [
    "I keep getting charged monthly even though I canceled.",
    "The mobile app freezes when I try to check out.",
    "My order was supposed to arrive last week.",
    "How do I get a receipt for my last purchase?",
    "The login page gives me a blank white screen.",
]

X_new = vectorizer.transform(new_tickets)
predictions = clf.predict(X_new)
probabilities = clf.predict_proba(X_new)

print("=== New Ticket Routing ===\n")
for ticket, pred, proba in zip(new_tickets, predictions, probabilities):
    confidence = max(proba) * 100
    print(f"  Ticket:     \"{ticket}\"")
    print(f"  Route to:   {pred.upper()} team (confidence: {confidence:.0f}%)\n")

> **Agentic use case**: A **ticket triage agent** uses this classifier to route tickets instantly (billing → billing team, technical → engineering, shipping → logistics). The LLM layer focuses on drafting personalized responses and summarizing — not the initial coarse classification.

## Understanding the Evaluation Metrics

| Metric | What It Measures | Formula |
|--------|-----------------|--------|
| **Precision** | Of the items the model *predicted* as class X, how many were actually X? | TP / (TP + FP) |
| **Recall** | Of the items that *actually are* class X, how many did the model find? | TP / (TP + FN) |
| **F1 Score** | Harmonic mean of precision and recall — a balanced single number | 2 × (P × R) / (P + R) |
| **Accuracy** | Overall fraction of correct predictions | Correct / Total |

- **High precision, low recall**: The model is cautious — when it predicts a class, it's usually right, but it misses some.
- **Low precision, high recall**: The model casts a wide net — it finds most examples, but has false positives.
- **F1** balances both — useful when you care equally about precision and recall.

## TF-IDF Vectorizer Options

The `TfidfVectorizer` has several important parameters:

In [ ]:
# Demonstrate the effect of different vectorizer settings
sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog chased the brown fox across the field.",
]

# Unigrams only (default)
v1 = TfidfVectorizer()
v1.fit(sample_texts)
print("Unigrams (default):")
print(f"  Features: {v1.get_feature_names_out().tolist()}\n")

# Unigrams + Bigrams
v2 = TfidfVectorizer(ngram_range=(1, 2))
v2.fit(sample_texts)
print("Unigrams + Bigrams:")
print(f"  Features: {v2.get_feature_names_out().tolist()}\n")

# With stop word removal
v3 = TfidfVectorizer(stop_words='english')
v3.fit(sample_texts)
print("With stop words removed:")
print(f"  Features: {v3.get_feature_names_out().tolist()}")

| Parameter | What It Does | Common Values |
|-----------|-------------|---------------|
| `stop_words='english'` | Removes common English words (the, is, at) | `'english'` or custom list |
| `ngram_range=(1, 2)` | Includes bigrams ("customer service", "not working") | `(1,1)` unigrams only, `(1,2)` uni+bi |
| `max_features=5000` | Limit vocabulary to top N features | Prevents memory issues on large corpora |
| `min_df=2` | Ignore terms appearing in fewer than N documents | Removes typos and ultra-rare words |
| `max_df=0.95` | Ignore terms appearing in more than 95% of documents | Removes corpus-wide common words |

---

# Part 4: Sentiment Analysis with VADER

## What Is VADER?

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based sentiment analysis tool specifically tuned for social media and short texts.

Unlike machine learning classifiers, VADER uses a **pre-built lexicon** — a dictionary of words with pre-assigned sentiment scores. It requires no training data.

Key properties:
- **Lexicon-based**: No training needed — works out of the box
- **Handles social media conventions**: Capitalization ("GREAT"), punctuation ("great!!!"), emojis, slang
- **Fast**: Microseconds per document
- **Lightweight**: No GPU, no model files

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Basic example
text = "The delivery was late but the support was excellent."
scores = sia.polarity_scores(text)

print(f"Text: \"{text}\"\n")
print("VADER Scores:")
for key, value in scores.items():
    print(f"  {key:<10} {value:+.4f}")

### Interpreting VADER Output

VADER returns four scores:

| Score | Range | Meaning |
|-------|-------|--------|
| `neg` | 0 to 1 | Proportion of text that is negative |
| `neu` | 0 to 1 | Proportion of text that is neutral |
| `pos` | 0 to 1 | Proportion of text that is positive |
| `compound` | -1 to +1 | **Overall sentiment** — the most useful single number |

**Compound score thresholds** (standard convention):
- `compound >= 0.05` → **Positive**
- `compound <= -0.05` → **Negative**
- `-0.05 < compound < 0.05` → **Neutral**

## Example 1: Sentiment Across Multiple Reviews

In [ ]:
reviews = [
    "Absolutely love this product! Best purchase ever.",
    "The product is okay, nothing special.",
    "Terrible experience. The item arrived broken.",
    "Great customer service but the product was mediocre.",
    "I hate this. Complete waste of money.",
    "Pretty good overall. Would recommend to friends.",
    "The delivery was late but the support was excellent.",
    "Not worth the price. Very disappointed.",
]

def classify_sentiment(compound):
    if compound >= 0.05:
        return "POSITIVE"
    elif compound <= -0.05:
        return "NEGATIVE"
    else:
        return "NEUTRAL"

print(f"{'Review':<55} {'Compound':>8}  {'Label'}")
print("-" * 80)
for review in reviews:
    scores = sia.polarity_scores(review)
    label = classify_sentiment(scores['compound'])
    print(f"{review:<55} {scores['compound']:>+8.4f}  {label}")

## Example 2: VADER Handles Intensity and Social Media Conventions

VADER is specifically designed to understand intensity signals that other models miss.

In [ ]:
# VADER understands capitalization, punctuation, and degree modifiers
intensity_examples = [
    ("The product is good.",           "baseline"),
    ("The product is GOOD.",           "capitalization = emphasis"),
    ("The product is good!",           "exclamation = emphasis"),
    ("The product is GOOD!!!",         "both = strong emphasis"),
    ("The product is really good.",    "degree modifier (really)"),
    ("The product is kind of good.",   "diminisher (kind of)"),
    ("The product is not good.",       "negation flips sentiment"),
    ("The product is not good at all.","strong negation"),
]

print(f"{'Text':<40} {'Compound':>8}  {'Note'}")
print("-" * 80)
for text, note in intensity_examples:
    score = sia.polarity_scores(text)['compound']
    print(f"{text:<40} {score:>+8.4f}  {note}")

Notice how the compound score increases with emphasis (capitalization, punctuation, "really"), decreases with diminishers ("kind of"), and flips with negation ("not"). This nuance is built into VADER's lexicon — no training required.

## Example 3: Real-Time Sentiment Monitoring

A practical agentic use case: monitoring a stream of customer messages and triggering alerts when sentiment drops below a threshold.

In [ ]:
# Simulated stream of customer messages
message_stream = [
    {"time": "10:01", "text": "Love the new update! Great improvements."},
    {"time": "10:03", "text": "The app is working smoothly today."},
    {"time": "10:05", "text": "Good experience overall."},
    {"time": "10:08", "text": "Something feels off with the checkout page."},
    {"time": "10:12", "text": "The app just crashed and I lost my cart."},
    {"time": "10:14", "text": "This is really frustrating! I can't complete my order."},
    {"time": "10:15", "text": "TERRIBLE experience. The whole site is broken!"},
    {"time": "10:18", "text": "Still can't check out. Very annoyed."},
    {"time": "10:22", "text": "Looks like it's fixed now. Thanks."},
    {"time": "10:25", "text": "Working again, happy to see the quick fix."},
]

ALERT_THRESHOLD = -0.3

print(f"{'Time':<8} {'Compound':>8}  {'Status':<10} {'Message'}")
print("-" * 85)

rolling_scores = []
for msg in message_stream:
    scores = sia.polarity_scores(msg['text'])
    compound = scores['compound']
    rolling_scores.append(compound)
    
    # Check for alert condition
    if compound < ALERT_THRESHOLD:
        status = "🚨 ALERT"
    else:
        status = "  OK"
    
    print(f"{msg['time']:<8} {compound:>+8.4f}  {status:<10} {msg['text']}")

# Summary statistics
avg_score = sum(rolling_scores) / len(rolling_scores)
alerts = sum(1 for s in rolling_scores if s < ALERT_THRESHOLD)
print(f"\n--- Summary ---")
print(f"Average sentiment: {avg_score:+.3f}")
print(f"Alerts triggered:  {alerts}")

> **Agentic use case**: VADER monitors the stream in real time. When sentiment drops below a threshold, it triggers an alert. An **LLM agent** can then summarize the main pain points ("checkout is broken — users are losing carts") and suggest actions.

---

# Part 5: Topic Modeling with LDA

## What Is Topic Modeling?

**Topic modeling** is an **unsupervised** technique that discovers latent themes (topics) in a collection of documents — no labels needed.

The most widely-used algorithm is **Latent Dirichlet Allocation (LDA)**.

### How LDA Works (Intuition)

LDA assumes each document is a **mixture of topics**, and each topic is a **mixture of words**:

```
Document: "My login failed and I couldn't reset my password"

  Topic distribution:  70% Login Issues, 20% Account Management, 10% Other

Where:
  Login Issues topic:      login, password, failed, error, reset, access
  Account Management topic: account, settings, profile, email, update
```

LDA learns both the topic-word distributions and the document-topic distributions from the data.

## The LDA Pipeline

```
Step 1: Preprocess — tokenize, remove stopwords, lemmatize
            ↓
Step 2: Build document-term matrix (bag of words or TF-IDF)
            ↓
Step 3: Fit LDA model — specify number of topics (k)
            ↓
Step 4: Inspect topic-word distributions to understand / label topics
```

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# A corpus of customer support tickets (unlabeled)
tickets = [
    # Login / authentication issues
    "I cannot log in to my account, the password reset is not working.",
    "My login keeps failing. I tried resetting the password three times.",
    "The login page shows an error when I enter my credentials.",
    "I am locked out of my account after too many failed login attempts.",
    "Password reset email never arrives in my inbox.",
    "Two-factor authentication code is not being accepted.",
    
    # Billing / payment issues
    "I was charged twice for the same subscription this month.",
    "My credit card was billed the wrong amount on my invoice.",
    "I need a refund for the duplicate charge on my account.",
    "The subscription price increased without any notification.",
    "I canceled but was still charged for the next billing cycle.",
    "The promo code did not apply the discount to my payment.",
    
    # Shipping / delivery issues
    "My package has not arrived and it has been two weeks since ordering.",
    "The delivery was delayed and the tracking number shows no updates.",
    "I received the wrong item in my shipment.",
    "The package arrived but the product inside was damaged.",
    "My order was marked delivered but I never received the package.",
    "Can I change the shipping address for a pending order?",
]

print(f"Corpus: {len(tickets)} tickets (no labels)")

In [ ]:
# Step 1-2: Build document-term matrix
# LDA typically works with raw counts (CountVectorizer), not TF-IDF
count_vectorizer = CountVectorizer(
    stop_words='english',
    max_df=0.9,   # ignore terms in >90% of docs
    min_df=2,     # ignore terms in <2 docs
)
doc_term_matrix = count_vectorizer.fit_transform(tickets)

print(f"Document-term matrix: {doc_term_matrix.shape[0]} documents × {doc_term_matrix.shape[1]} terms")

# Step 3: Fit LDA with 3 topics
n_topics = 3
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
)
lda.fit(doc_term_matrix)

print(f"LDA model fitted with {n_topics} topics.")

In [ ]:
# Step 4: Inspect topic-word distributions
feature_names = count_vectorizer.get_feature_names_out()

def display_topics(model, feature_names, n_top_words=8):
    """Display the top words for each topic."""
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        weights = [topic[i] for i in top_indices]
        
        print(f"\nTopic {topic_idx + 1}:")
        for word, weight in zip(top_words, weights):
            bar = "█" * int(weight / max(weights) * 20)
            print(f"  {word:<18} {bar}")

display_topics(lda, feature_names)

### Interpreting the Topics

Look at the top words for each topic and assign a human-readable label:

- **Topic with** login, password, account, reset, failed → **"Authentication Issues"**
- **Topic with** charged, subscription, billing, refund, payment → **"Billing Problems"**
- **Topic with** package, delivery, shipped, arrived, order → **"Shipping & Delivery"**

LDA discovered the natural groupings in the data — without any labels!

In [ ]:
# See the topic distribution for each document
doc_topic_distributions = lda.transform(doc_term_matrix)

print(f"{'Ticket (first 55 chars)':<58} {'T1':>5} {'T2':>5} {'T3':>5}  {'Dominant'}")
print("-" * 90)
for i, (ticket, dist) in enumerate(zip(tickets, doc_topic_distributions)):
    dominant = f"Topic {dist.argmax() + 1}"
    print(f"{ticket[:55]:<58} {dist[0]:>5.2f} {dist[1]:>5.2f} {dist[2]:>5.2f}  {dominant}")

Each ticket has a probability distribution over topics. Most tickets are strongly associated with one dominant topic, but some may be a mixture — e.g., a ticket about both billing and account access.

## Using LDA Topics as Features

Once you've discovered topics, the topic distribution for each document becomes a **compact feature vector** you can use for downstream tasks.

In [ ]:
# Classify a new ticket using the trained topic model
new_tickets = [
    "My password stopped working and I can't access my account.",
    "I was overcharged on my last bill.",
    "The shipment is stuck in transit for over a week.",
]

new_dtm = count_vectorizer.transform(new_tickets)
new_topics = lda.transform(new_dtm)

topic_labels = ["Authentication", "Billing", "Shipping"]  # based on our inspection above

print("New ticket topic assignments:\n")
for ticket, dist in zip(new_tickets, new_topics):
    dominant_idx = dist.argmax()
    print(f"  \"{ticket}\"")
    print(f"  → {topic_labels[dominant_idx]} (confidence: {dist[dominant_idx]:.1%})\n")

> **Agentic use case**: Run LDA over thousands of unlabeled customer tickets. Discover that topics correspond to "login issues", "billing problems", "shipping delays". Use these as features for dashboards and alerts. An LLM agent can then **label** the topics ("Topic 1 = Authentication Issues") or **explain** them to stakeholders in natural language.

---

# Part 6: Comparing All Three Approaches

Let's see how the same text is analyzed differently by each technique.

In [ ]:
test_text = "I was charged twice for a subscription I already canceled. The support team was unhelpful."

print(f"Text: \"{test_text}\"\n")
print("=" * 70)

# 1. Supervised Classification (from our ticket triage model)
X_test_vec = vectorizer.transform([test_text])
pred = clf.predict(X_test_vec)[0]
proba = clf.predict_proba(X_test_vec)[0]
print(f"\n1. SUPERVISED CLASSIFICATION (Ticket Triage)")
print(f"   Prediction: {pred.upper()}")
for cls, p in zip(clf.classes_, proba):
    print(f"     {cls:<12} {p:.1%}")

# 2. Sentiment Analysis (VADER)
scores = sia.polarity_scores(test_text)
sentiment = classify_sentiment(scores['compound'])
print(f"\n2. SENTIMENT ANALYSIS (VADER)")
print(f"   Compound score: {scores['compound']:+.4f} → {sentiment}")

# 3. Topic Model (LDA)
test_dtm = count_vectorizer.transform([test_text])
test_topics = lda.transform(test_dtm)[0]
dominant_topic = test_topics.argmax()
print(f"\n3. TOPIC MODEL (LDA)")
print(f"   Dominant topic: Topic {dominant_topic + 1} ({topic_labels[dominant_topic]})")
for i, (label, score) in enumerate(zip(topic_labels, test_topics)):
    print(f"     {label:<18} {score:.1%}")

### When to Use Each

| Technique | Type | Needs Labels? | Best For |
|-----------|------|--------------|----------|
| **Supervised classification** | Supervised | Yes | Routing to known, stable categories |
| **VADER sentiment** | Lexicon-based | No | Real-time sentiment monitoring |
| **LDA topic modeling** | Unsupervised | No | Discovering unknown themes in large corpora |

---

# Part 7: The Big Picture — Classical NLP in Agentic Systems

```
┌─────────────────────────────────────────────────────────────────────────┐
│               CUSTOMER INTELLIGENCE AGENT SYSTEM                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  INCOMING DATA: tickets, reviews, social media, emails                  │
│                          ↓                                              │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │  CLASSICAL NLP LAYER (fast, free, deterministic)             │       │
│  │                                                              │       │
│  │  • TF-IDF + Classifier → route to billing / tech / shipping  │       │
│  │  • VADER → real-time sentiment score + alert triggers        │       │
│  │  • LDA → discover emerging themes in unlabeled data          │       │
│  └──────────────────────────────────────────────────────────────┘       │
│                          ↓                                              │
│  STRUCTURED SIGNALS: category, sentiment, topic, confidence             │
│                          ↓                                              │
│  ┌──────────────────────────────────────────────────────────────┐       │
│  │  LLM AGENT LAYER (reasoning, generation, nuance)             │       │
│  │                                                              │       │
│  │  • Draft personalized responses                              │       │
│  │  • Summarize weekly sentiment trends                         │       │
│  │  • Label discovered topics in natural language               │       │
│  │  • Handle edge cases and escalations                         │       │
│  └──────────────────────────────────────────────────────────────┘       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

# Practice Exercises

Try these exercises to reinforce what you've learned.

## Exercise 1: Build a Spam Classifier

Using the training data below, build a TF-IDF + Logistic Regression classifier that distinguishes spam from legitimate messages. Then classify the test messages.

In [ ]:
train_texts = [
    "Congratulations! You've won a free iPhone. Click here to claim.",
    "URGENT: Your bank account has been compromised. Verify now.",
    "Make $5000 per week working from home. Limited time offer!",
    "You have been selected for a free vacation. Act now!",
    "Buy cheap medications online. No prescription needed.",
    "Hi Peter, can we schedule our meeting for Thursday at 2pm?",
    "The quarterly report is attached. Please review by Friday.",
    "Team lunch tomorrow at noon in the main conference room.",
    "I've pushed the code changes to the dev branch for review.",
    "Reminder: project deadline is next Monday.",
]
train_labels = ["spam", "spam", "spam", "spam", "spam",
                "legit", "legit", "legit", "legit", "legit"]

test_texts = [
    "You've been selected for an exclusive cash prize!",
    "Can you send me the updated spreadsheet?",
    "FREE gift cards available. Click the link below.",
    "The design review meeting has been moved to Friday.",
]

# TODO: Build a TF-IDF + Logistic Regression classifier
# Then predict and print results for the test messages

# Your code here

## Exercise 2: Sentiment Trend Analysis

Use VADER to analyze the sentiment of each review below. Calculate the average sentiment per product and identify which product has the most negative feedback.

In [ ]:
product_reviews = {
    "Widget Pro": [
        "Excellent build quality, very satisfied.",
        "Good product but the manual is confusing.",
        "Fantastic! Best widget I've ever used.",
    ],
    "Widget Lite": [
        "Cheap and flimsy, broke after a week.",
        "Not worth the money. Very disappointed.",
        "It's okay for the price, nothing special.",
    ],
    "Widget Max": [
        "Absolutely love it! Premium feel.",
        "Great product but overpriced.",
        "The battery life is amazing.",
    ],
}

# TODO:
# 1. Compute VADER compound score for each review
# 2. Calculate average sentiment per product
# 3. Identify which product has the lowest average sentiment

# Your code here

## Exercise 3: Topic Discovery

Run LDA with **k=2** topics on the documents below. Inspect the top words and assign human-readable labels to each topic.

In [ ]:
documents = [
    "The stock market rallied on strong earnings reports from tech companies.",
    "Investors are optimistic about interest rate cuts boosting the economy.",
    "Bond yields fell as inflation data came in below expectations.",
    "The central bank announced new monetary policy measures.",
    "The team won the championship after a dramatic overtime victory.",
    "The star player signed a record-breaking contract extension.",
    "Fans celebrated the playoff win with a downtown parade.",
    "The coach was named manager of the year after the winning season.",
]

# TODO:
# 1. Build a CountVectorizer and document-term matrix
# 2. Fit LDA with n_components=2
# 3. Display top words per topic
# 4. Assign human-readable labels

# Your code here

---

## Summary

| Technique | Type | Library | Best For |
|-----------|------|---------|----------|
| **TF-IDF + Classifier** | Supervised | scikit-learn | Routing to known categories with labeled data |
| **VADER** | Lexicon-based | NLTK | Real-time sentiment monitoring, no training needed |
| **LDA** | Unsupervised | scikit-learn | Discovering unknown themes in unlabeled corpora |

### Key Takeaways

1. **TF-IDF** converts text to numbers by weighing words that are common in a document but rare across the corpus
2. **Supervised classification** (TF-IDF + Logistic Regression) is fast, cheap, deterministic, and explainable — inspect coefficients to see what the model learned
3. **VADER** provides instant sentiment scoring with no training data, and handles social media conventions (caps, punctuation, negation)
4. **LDA** discovers latent topics in unlabeled data — use the top words per topic to assign human-readable labels
5. In agentic systems, classical NLP handles **high-volume, low-cost classification and monitoring**, while LLMs handle **reasoning, generation, and edge cases**